# 06. Structured Output in LangChain 📐

Structured output forces models and agents to return data in a specific, predictable schema (e.g. Pydantic models, TypedDicts, Dataclasses) instead of raw text.

### 1. Environment Setup & Model Initialization

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq

load_dotenv()

# Initialize model
llm = ChatGroq(model="qwen/qwen3.8-27b")

### 2. Standalone Model (`with_structured_output`)
- Uses `.with_structured_output(Schema)` to parse responses directly into typed objects.

In [2]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age of the person")
    mobile_no: str = Field(description="Contact mobile number")
    email: str = Field(description="Email address")

# Bind structured output schema
structured_llm = llm.with_structured_output(ContactInfo)

response = structured_llm.invoke(
    "Name: Viraj Ladvikar, age: 27, contact: 7304549807, email: viraj@gmail.com"
)

print(response)
print("Name:", response.name)
print("Email:", response.email)
print("Mobile:", response.mobile_no)

name='Viraj Ladvikar' age=27 mobile_no='7304549807' email='viraj@gmail.com'
Name: Viraj Ladvikar
Email: viraj@gmail.com
Mobile: 7304549807


### 3. Nested Structured Schema
- Define complex nested Pydantic models with lists and child objects.

In [3]:
class Actor(BaseModel):
    name: str = Field(description="Name of the actor")
    role: str = Field(description="Character role played by the actor")

class MovieDetails(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year")
    director: str = Field(description="Director of the movie")
    cast: list[Actor] = Field(description="List of main actors")
    genres: list[str] = Field(description="List of movie genres")

movie_llm = llm.with_structured_output(MovieDetails)

movie_response = movie_llm.invoke("Provide details about the movie Inception directed by Christopher Nolan.")
print(movie_response)

title='Inception' year=2010 director='Christopher Nolan' cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Miles')] genres=['Action', 'Sci-Fi', 'Thriller']


### 4. Agent Structured Output (`create_agent` with `response_format`)
- Passes the schema to `create_agent`, and the result is available in `result["structured_response"]`.

In [4]:
from langchain.agents import create_agent

# 1. Define schema
class ApplicantInfo(BaseModel):
    name: str = Field(description="Full name of the applicant")
    skills: list[str] = Field(description="Technical skills of the applicant")
    experience_years: int = Field(description="Total years of work experience")
    expected_salary: int = Field(description="Expected annual salary in INR")

# 2. Create agent with response_format
agent = create_agent(
    model=llm,
    tools=[],
    response_format=ApplicantInfo
)

# 3. Invoke agent
agent_response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract applicant details: Rahul Sharma has 4 years of experience in Python, SQL, and API testing. His expected salary is 1200000 INR per year."
    }]
})

# 4. Access structured response
result = agent_response["structured_response"]
print("Result:", result)
print("Applicant Name:", result.name)
print("Skills:", result.skills)
print("Experience:", result.experience_years, "years")

Result: name='Rahul Sharma' skills=['Python', 'SQL', 'API testing'] experience_years=4 expected_salary=1200000
Applicant Name: Rahul Sharma
Skills: ['Python', 'SQL', 'API testing']
Experience: 4 years


### 5. Using `TypedDict`
- Lightweight dictionary-based schema without runtime validation overhead.

In [5]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

typed_llm = llm.with_structured_output(MovieDict)
dict_response = typed_llm.invoke("Please provide details for the movie The Matrix.")
print(dict_response)

{'director': 'The Wachowskis', 'rating': 8.7, 'title': 'The Matrix', 'year': 1999}


### 6. Using Python `dataclasses`
- Python standard dataclasses for clean object representation.

In [6]:
from dataclasses import dataclass

@dataclass
class PersonProfile:
    """Information about a person."""
    name: str
    email: str
    city: str

dataclass_agent = create_agent(
    model=llm,
    tools=[],
    response_format=PersonProfile
)

dataclass_res = dataclass_agent.invoke({
    "messages": [{"role": "user", "content": "John Doe lives in Mumbai and his email is john@example.com"}]
})

print(dataclass_res["structured_response"])

PersonProfile(name='John Doe', email='john@example.com', city='Mumbai')
